In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -q scikit-learn==1.5.2

In [ ]:
import sklearn
sklearn.__version__

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="seaborn")
warnings.filterwarnings("ignore", category=FutureWarning, module="seaborn")

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import root_mean_squared_log_error,mean_squared_error, mean_absolute_error, r2_score

import optuna
import lightgbm as lgb

import torch
from sklearn.pipeline import Pipeline

In [ ]:
train_df=pd.read_csv('/kaggle/input/playground-series-s4e12/train.csv')
test_df=pd.read_csv('/kaggle/input/playground-series-s4e12/test.csv')

# Exploratory Data Analysis (EDA)

Exploratory Data Analysis (EDA) is a critical step in any data project. It allows us to understand, summarize, and visualize the dataset effectively, paving the way for further analysis or modeling.

---

### 🎯 Objectives

- 🕵️‍♀️ Gain insights into the data.
- 📊 Visualize distributions, relationships, and patterns.
- 🧹 Identify missing values, outliers, and data inconsistencies.etection.

---

Let’s dive into the EDA! 🚀


## 📜 Dataset Overview

In [ ]:
# Check dataset shape and first rows
print(f"Dataset contains {train_df.shape[0]} rows and {train_df.shape[1]} columns.")
train_df.head()


In [ ]:
train_df.info()

In [ ]:
# Save 'id' column for submission
test_ids = test_df['id']

# Define the target column
target_column = 'Premium Amount'

# Select categorical and numerical columns (initial)
categorical_columns = train_df.select_dtypes(include=['object']).columns
numerical_columns = train_df.select_dtypes(exclude=['object']).columns

# Print out column information
print("Target Column:", target_column)
print("\nCategorical Columns:", categorical_columns.tolist())
print("\nNumerical Columns:", numerical_columns.tolist())

## 📊 Descriptive Statistics

In [ ]:
train_df.describe().round(2)

In [ ]:
for column in categorical_columns:
    num_unique = train_df[column].nunique()
    print(f"'{column}' has {num_unique} unique categories.")

In [ ]:
# Print top 10 unique value counts for each categorical column
for column in categorical_columns:
    print(f"\nTop value counts in '{column}':\n{train_df[column].value_counts().head(10)}")

In [ ]:
print("The mean of columns:")
print(train_df[numerical_columns].mean())

print("\nThe std dev of columns:")
print(train_df[numerical_columns].std())

print("\nThe skewness of columns:")
print(train_df[numerical_columns].skew())

## 🧹 Data Cleaning Insights

In [ ]:
plt.figure(figsize=(15,9))
plt.title("Visualizing Missing Values")
sns.heatmap(train_df.isnull(), cbar=False, cmap=sns.color_palette('magma'), yticklabels=False);
plt.show()

## 🖼️ Visual Exploration

In [ ]:
# Create a color palette for the columns
palette = sns.color_palette('tab10', len(numerical_columns))
color_dict = dict(zip(numerical_columns, palette))

# Create a grid of subplots for histograms, boxplots, and scatterplots/violin plots
fig = plt.figure(figsize=(30, 10 * len(numerical_columns)))
gs = gridspec.GridSpec(2 * len(numerical_columns), 2, figure=fig)

df_binned = train_df.copy()

for i, column in enumerate(numerical_columns):

    if train_df[column].nunique() > 50: discrete = False
    else : discrete = True
    
    # Plot histogram with a unique color
    ax_hist = fig.add_subplot(gs[2 * i, 0])
    sns.histplot(
        data=train_df, x=column, fill=True, common_norm=False, alpha=0.6,
        linewidth=0.8, color=color_dict[column], ax=ax_hist,  discrete = discrete
    )
    
    # Plot boxplot with the same unique color
    ax_box = fig.add_subplot(gs[2 * i + 1, 0])
    sns.boxplot(data=train_df, x=column, ax=ax_box, color=color_dict[column])
    ax_box.set_title(f'{column} vs Target (Boxplot)', fontsize=14)
    sns.despine(ax=ax_box)

    # Conditional plot: violin plot or barplot based on unique values, fallback to scatterplot
    ax_conditional = fig.add_subplot(gs[2 * i:2 * i + 2, 1])  # Merges 2 rows
    if train_df[column].nunique() <= 10:
        # If the column has 10 or fewer unique values, use a violin plot
        sns.violinplot(data=train_df, x=column, y=target_column, ax=ax_conditional, color=color_dict[column], alpha=0.6)
        ax_conditional.set_title(f'{column} vs {target_column} (Violin Plot)', fontsize=14)
    else:
        # Bin the column into 10 intervals, but keep original target column values
        df_binned['Binned Column'] = pd.cut(train_df[column], bins=10)
        sns.violinplot(data=df_binned, x='Binned Column', y=target_column, ax=ax_conditional, color=color_dict[column], alpha=0.6)
        ax_conditional.set_title(f'{column} (Binned) vs {target_column} (Violin Plot)', fontsize=14)
        ax_conditional.set_xlabel(f'{column} (Binned)', fontsize=12)

plt.tight_layout()  # Adjust subplots to fit into the figure area
plt.show()

In [ ]:
# Filtrer les colonnes catégorielles et exclure 'Policy Start Date'
filtered_columns = [col for col in categorical_columns if col != 'Policy Start Date']

# Créer des sous-graphiques pour barplots et boxplots
fig, axes = plt.subplots(len(filtered_columns), 2, figsize=(15, 5 * len(filtered_columns)))

for i, column in enumerate(filtered_columns):
    # Barplot à gauche
    sns.countplot(data=train_df, x=column, ax=axes[i, 0], palette='tab10')
    axes[i, 0].set_title(f'Distribution of {column}', fontsize=14)
    axes[i, 0].set_xlabel(column, fontsize=12)
    axes[i, 0].set_ylabel('Count', fontsize=12)
    sns.despine(ax=axes[i, 0])

    # Boxplot à droite
    sns.boxplot(data=train_df, x=column, y=target_column, ax=axes[i, 1], palette='tab10')
    axes[i, 1].set_title(f'{column} vs {target_column}', fontsize=14)
    axes[i, 1].set_xlabel(column, fontsize=12)
    axes[i, 1].set_ylabel(target_column, fontsize=12)
    sns.despine(ax=axes[i, 1])

plt.tight_layout()  # Ajustement global des sous-graphiques
plt.show()


In [ ]:
# Calculate the correlation matrix
correlation_matrix = train_df[numerical_columns].corr()

# Plot the heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, linewidths=0.5)
plt.title("Correlation Heatmap of Numerical Variables", fontsize=16)
plt.show()


### 🏁 Next Steps

- Dive deeper into specific features of interest.
- Address missing values and.
- Prepare the dataset for modeling.

# Data Preprocessing

Data preprocessing is a crucial step in preparing the dataset for analysis and modeling. It ensures the data is clean, consistent, and ready for machine learning algorithms.

---

### 🔍 Objectives

- Handle missing values.
- Encode categorical features.
- Standardize or normalize numerical features.
- Create new features or transform existing ones if necessary.

## 🗓️ Feature Transformation: Date Handling

In this step, we transform the `Policy Start Date` feature into multiple useful date-related features. This helps to extract temporal patterns and improve the model's ability to learn from the data.

In [ ]:
def date(df):

    df['Policy Start Date'] = pd.to_datetime(df['Policy Start Date'])
    df['Year'] = df['Policy Start Date'].dt.year
    df['Day'] = df['Policy Start Date'].dt.day
    df['Month'] = df['Policy Start Date'].dt.month
    df['Month_name'] = df['Policy Start Date'].dt.month_name()
    df['Day_of_week'] = df['Policy Start Date'].dt.day_name()
    df['Week'] = df['Policy Start Date'].dt.isocalendar().week
    df['Year_sin'] = np.sin(2 * np.pi * df['Year'])
    df['Year_cos'] = np.cos(2 * np.pi * df['Year'])
    min_year = df['Year'].min()
    max_year = df['Year'].max()
    df['Year_sin'] = np.sin(2 * np.pi * (df['Year'] - min_year) / (max_year - min_year))
    df['Year_cos'] = np.cos(2 * np.pi * (df['Year'] - min_year) / (max_year - min_year))
    df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12) 
    df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)
    df['Day_sin'] = np.sin(2 * np.pi * df['Day'] / 31)  
    df['Day_cos'] = np.cos(2 * np.pi * df['Day'] / 31)
    df['Group']=(df['Year']-2020)*48+df['Month']*4+df['Day']//7
    
    df.drop('Policy Start Date', axis=1, inplace=True)

    return df

# Apply the date function to both datasets
train_df = date(train_df)
test_df = date(test_df)

In [ ]:
# Define features and target
numerical_features = [
    'Age', 'Annual Income', 'Number of Dependents', 'Health Score', 
    'Previous Claims', 'Vehicle Age', 'Credit Score', 'Insurance Duration', 
    'Year_sin', 'Year_cos', 'Month_sin', 'Month_cos', 'Day_sin', 'Day_cos'
]
categorical_features = [
    'Gender', 'Marital Status', 'Education Level', 'Occupation', 'Location',
    'Policy Type', 'Customer Feedback', 'Smoking Status', 'Exercise Frequency', 
    'Property Type', 'Month_name', 'Day_of_week'
]
target_column = 'Premium Amount'

## ✂️ Splitting Data: Features and Target

In this step, we split the training dataset into:
- **Features (`X`)**: The independent variables used to predict the target.
- **Target (`y`)**: The dependent variable that the model will learn to predict.

In [ ]:
# Split train data into features and target
X = train_df.drop(columns=[target_column, 'id', 'Group', 'Year', 'Month', 'Day', 'Week'])
y = train_df[target_column]

## 🛠️ Handle Missing Values & Preprocessing Pipeline

In this step, we handle missing values and set up a preprocessing pipeline to prepare the data for machine learning.

---

### 🔍 What We Did

1. **Missing Values Imputation**:
   - **Numerical Features**:
     - Replaced missing values with the **mean** of the respective columns.
   - **Categorical Features**:
     - Replaced missing values with the constant value **"Unknown"**.

2. **Feature Scaling & Encoding**:
   - **Numerical Features**:
     - Standardized using `StandardScaler` to normalize the values.
   - **Categorical Features**:
     - One-Hot Encoded to handle categorical variables as numerical inputs.

3. **Combined Using a ColumnTransformer**:
   - Applied preprocessing selectively to numerical and categorical features using a single, unified pipeline.

In [ ]:
# Preprocessing pipeline for numerical features
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    #('scaler', StandardScaler())                       # Scale numerical features
])

# Preprocessing pipeline for categorical features
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),  # Handle missing values
    ('onehot', OneHotEncoder(handle_unknown='ignore'))                      # Encode categorical features
])

# Combine pipelines into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numerical_features),
        ('cat', cat_pipeline, categorical_features)
    ]
)

# Preprocess train and test data
X_processed = preprocessor.fit_transform(X)
test_processed = preprocessor.transform(test_df.drop(columns=['id', 'Group', 'Year', 'Month', 'Day', 'Week']))

## ✂️ Splitting Data: Training and Validation Sets

Here, we split the preprocessed data into training and validation sets to evaluate the model's performance during training.

In [ ]:
# Split the data
X_train, X_val, y_train, y_val = train_test_split(X_processed, y, test_size=0.2, random_state=42)

# Model Training

Training the model is the core step in any machine learning pipeline. Here, we use the processed features and target variable to fit a predictive model and evaluate its performance on a validation set.

---

### 🔍 Objectives
- Train the model using the training dataset (`X_train`, `y_train`).
- Evaluate the model on the validation set (`X_val`, `y_val`).
- Optimize the model's parameters to improve its performance.

---


## 🔧 Hyperparameter Optimization with Optuna

Optuna is a powerful library for hyperparameter optimization. In this step, we use Optuna to fine-tune the hyperparameters of a LightGBM model to achieve optimal performance.

In [ ]:
# Define Optuna optimization function
def objective(trial):
    # Define parameter search space
    param = {
        "objective": "regression",
        "metric": "rmse",
        "boosting_type": trial.suggest_categorical("boosting_type", ["gbdt", "dart"]),
        "num_leaves": trial.suggest_int("num_leaves", 200, 512),
        "learning_rate": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),
        "feature_fraction": trial.suggest_uniform("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_uniform("bagging_fraction", 0.6, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 5, 12),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 100),
        "max_depth": trial.suggest_int("max_depth", -1, 16),  # -1 means no limit
        "lambda_l1": trial.suggest_loguniform("lambda_l1", 1e-4, 10.0),
        "lambda_l2": trial.suggest_loguniform("lambda_l2", 1e-4, 10.0),
        "device_type": "gpu",  # Enable GPU support
        "seed" : 42

    }

    # Create a LightGBM dataset
    dtrain = lgb.Dataset(X_train, label=y_train)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    # Train LightGBM model
    model = lgb.train(
        param,
        dtrain,
        valid_sets=[dval],
    )

    # Predict on validation set
    y_val_pred = model.predict(X_val)
    
    # Compute RMSLE using sklearn's root_mean_squared_log_error
    rmsle = root_mean_squared_log_error(y_val, np.maximum(y_val_pred, 0))
    return rmsle

# Run Optuna study
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=1)

### 🔧 Best Hyperparameters

After running the hyperparameter optimization process with Optuna, the best combination of parameters was identified. These parameters will be used to train the final LightGBM model.

In [ ]:
# Initialize or update the best_params dictionary
best_params = {
    'boosting_type': 'dart',
    'num_leaves': 384,
    'learning_rate': 0.024680120465142227,
    'feature_fraction': 0.9883068358315126,
    'bagging_fraction': 0.7201712704805496,
    'bagging_freq': 7,
    'min_data_in_leaf': 50,
    'max_depth': 15,
    'lambda_l1': 0.0011290211269753322,
    'lambda_l2': 3.056310541294088,
    'seed': 42
}

## 🚀 Train Final Model with Best Parameters

Finally, we train the final LightGBM model using the optimal hyperparameters identified during the hyperparameter optimization process. This ensures the model is trained with the most effective configuration for achieving the best performance.

In [ ]:
# Train final model with best parameters
#best_params = study.best_params

final_model = lgb.train(
    best_params,
    lgb.Dataset(X_processed, label=y),
)

## 📊 Model Evaluation

In this section, we evaluate the performance of the trained model using multiple metrics and visualizations. This helps us understand how well the model performs and identify areas for improvement.

---

### 🔍 Performance Metrics
We compute the following metrics to evaluate the model:
- **RMSLE (Root Mean Squared Logarithmic Error)**: Measures the ratio-based prediction error, suitable for skewed datasets.
- **RMSE (Root Mean Squared Error)**: Quantifies the average magnitude of prediction errors.
- **MAE (Mean Absolute Error)**: Represents the average absolute difference between predicted and actual values.
- **R² (Coefficient of Determination)**: Indicates the proportion of variance in the target explained by the model.
- **MAPE (Mean Absolute Percentage Error)**: Expresses prediction errors as a percentage of actual values.

In [ ]:
# 1. Performance Metrics
y_pred = final_model.predict(X_processed)

# Calcul des métriques
rmsle = root_mean_squared_log_error(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
mae = mean_absolute_error(y, y_pred)
r2 = r2_score(y, y_pred)
mape = np.mean(np.abs((y - y_pred) / y)) * 100

# Display performance metrics
print(f"\nPerformance Metrics:\n{'-'*30}")
print(f"RMSLE: {rmsle:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R²: {r2:.4f}")
print(f"MAPE: {mape:.2f}%")

# 2. Feature Importance
importances = final_model.feature_importance(importance_type='split')  # or 'gain'
features = preprocessor.get_feature_names_out()
sorted_indices = importances.argsort()[::-1]

# Create a DataFrame for feature importances
importance_df = pd.DataFrame({
    'Feature': [features[i] for i in sorted_indices],
    'Importance': importances[sorted_indices]
})

# Plot top 10 feature importances
plt.figure(figsize=(12, 6))
sns.barplot(data=importance_df.head(10), x='Importance', y='Feature', palette="coolwarm")
plt.title("Top 10 Feature Importances", fontsize=16, fontweight='bold')
plt.xlabel("Feature Importance", fontsize=12)
plt.ylabel("Feature", fontsize=12)
plt.tight_layout()
plt.show()

# 3. Residual Analysis
residuals = y - y_pred

# Residuals vs Predicted Values
plt.figure(figsize=(12, 6))
sns.scatterplot(x=y_pred, y=residuals, alpha=0.6, color="#007acc")
plt.axhline(y=0, color='red', linestyle='--', linewidth=1.5)
plt.title("Residuals vs Predicted Values", fontsize=16, fontweight='bold')
plt.xlabel("Predicted Values", fontsize=12)
plt.ylabel("Residuals", fontsize=12)
plt.tight_layout()
plt.show()

# Residual Distribution
plt.figure(figsize=(10, 6))
sns.histplot(residuals, bins=30, kde=True, color="#55a630")
plt.axvline(x=0, color='red', linestyle='--', linewidth=1.5)
plt.title("Distribution of Residuals", fontsize=16, fontweight='bold')
plt.xlabel("Residuals", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.tight_layout()
plt.show()

before KNN imputer :

___

Performance Metrics:
- RMSLE: 1.0634
- RMSE: 906.9466
- MAE: 625.4777
- R²: -0.0993
- MAPE: 204.97%

# Generate Predictions & Prepare Submission

Finally, we use the trained model to predict outcomes on the test set and format the results into a submission file for the competition.

In [ ]:
# Make predictions on the test set
test_predictions = final_model.predict(test_processed, num_iteration=final_model.best_iteration)

# Prepare submission file
submission = pd.DataFrame({'id': test_df['id'], 'Premium Amount': test_predictions})
submission.to_csv("submission.csv", index=False)

# Conclusion

Finally, we have completed the end-to-end process of building and evaluating a machine learning model. This journey included data exploration, preprocessing, model training, hyperparameter optimization, and preparing the final submission file.

---

### 🔑 Key Highlights

1. **Exploratory Data Analysis (EDA)**:
   - Gained valuable insights into the dataset through visualization and descriptive statistics.
   - Identified key patterns, correlations, and potential data quality issues.

2. **Data Preprocessing**:
   - Handled missing values, encoded categorical features, and standardized numerical features.
   - Engineered new features, especially time-based features, to improve the model's predictive capabilities.

3. **Model Training and Evaluation**:
   - Trained a LightGBM model with optimized hyperparameters using Optuna.
   - Evaluated the model's performance using metrics such as RMSE, RMSLE, and \( R^2 \).
   - Performed residual analysis to validate the model's behavior and assumptions.

4. **Final Submission**:
   - Used the trained model to predict outcomes on the test dataset.
   - Prepared and saved the submission file in the required format.

---

### 🚀 Next Steps

- Explore additional feature engineering opportunities to further enhance model performance.
- Experiment with ensemble methods or other advanced algorithms for potential improvements.
- Analyze and fine-tune the model using feedback from the competition leaderboard.

---

Thank you for exploring this notebook!
